In [1]:
"""
- Reads:
    SNP+Protein Folds.csv
    HealthMetrics+MentalHealth+DietNutrient.csv

- Produces:
    Final_Dataset.csv

Behavior:
1. Detect unique SEQN in Health file (keeps first occurrence for mapping).
2. Assigns those SEQN values to Genomes(SNP+Protein Folds) file (by index order).
3. Merges Genomes info back into the full table (left join on SEQN).
4. Saves output

If counts don't match, the script will show helpful error/info and won't overwrite files.
"""

import pandas as pd
import sys

# --- Paths (change if needed) ---
GENOME_PATH = "Snp_Seqs+Protein_Seq_Folds.csv"
HEALTH_PATH = "HealthMetrics+MentalHealth+DietNutrient.csv"
OUTPUT_PATH = "Final_Dataset.csv"

def main():
    # Load files
    print("Loading files...")
    genome_df = pd.read_csv(GENOME_PATH)
    health_df = pd.read_csv(HEALTH_PATH)

    print(f"Genome rows: {len(genome_df)}")
    print(f"Health rows: {len(health_df)}")
    if "SEQN" not in health_df.columns:
        raise KeyError("Health metrics CSV does not contain 'SEQN' column. Please check.")

    # Count unique SEQN values
    unique_seqn = health_df['SEQN'].nunique()
    print(f"Unique SEQN in health file: {unique_seqn}")

    # If genome rows != unique SEQN, stop with helpful message
    if len(genome_df) != unique_seqn:
        msg = (
            f"WARNING: genome rows ({len(genome_df)}) != unique SEQN ({unique_seqn}).\n"
            "This script expects one genome row per unique SEQN so we can map 1:1.\n"
            "Options:\n"
            " - If genome rows < unique SEQN: provide more genome rows or reduce SEQN.\n"
            " - If genome rows > unique SEQN: provide which SEQNs to map or reduce genome rows.\n"
            "Current behavior: if lengths mismatch, the script will attempt to match up to the\n"
            "minimum of the two lengths, and issue a warning. You can edit the script to\n"
            "change behavior to strict error instead."
        )
        print(msg)

    # Create deduplicated health list (first occurrence) to map SEQNs -> genome rows
    health_unique = health_df.drop_duplicates(subset="SEQN", keep="first").reset_index(drop=True)
    print(f"Health unique rows after drop_duplicates: {len(health_unique)}")

    # Determine mapping length to use
    n_map = min(len(genome_df), len(health_unique))
    if n_map != len(genome_df) or n_map != len(health_unique):
        print(f"Partial mapping will use first {n_map} rows of each file (truncating extras).")

    # Truncate both to n_map for 1:1 mapping
    genome_map = genome_df.iloc[:n_map].copy().reset_index(drop=True)
    health_map = health_unique.iloc[:n_map].copy().reset_index(drop=True)

    # Assign SEQN from health_map to genome_map
    genome_map['SEQN'] = health_map['SEQN'].values
    # If genome_df has an existing seq id column (e.g., seq_id) keep it
    # Now merge genome_map with full health_df
    print("Merging genome info back into the full health dataframe (left join on SEQN)...")
    # If there were more genome rows than health_unique, those extras were dropped; if fewer, some SEQNs will get NaN genome info.
    merged = pd.merge(health_df, genome_map, on='SEQN', how='left', suffixes=('', '_genome'))

    # Summary counts
    mapped_count = merged['Seq_Id'].notna().sum() if 'Seq_Id' in merged.columns else merged[['Snp_Sequence','Protein_Sequence_Folds']].notna().any(axis=1).sum()
    total_rows = len(merged)
    print(f"Merged rows: {total_rows}")
    print(f"Rows with genome info present (approx): {mapped_count}")
    if mapped_count < total_rows:
        print("WARNING: Some health rows do not have genome info (this is expected if health has multiple rows per SEQN).")
    
    # Removing SEQN & Snp_Id Col (Not required once merged)
    merged = merged.drop(['SEQN','Seq_Id'], axis=1)
    # Re-arrange of cols
    merged = merged[['Gender','Age','BMI','Systolic_bp','Diastolic_bp','Cholesterol','Diabetes',
                     'Stress','Sleep','Mood',
                     'Snp_Sequence','Protein_Sequence_Folds',
                     'Occasion','DietType','FoodDescription',
                     'Grams','Energy','Proteins','Carbs','Sugars','Fiber','Fat','Cholestrol'
                    ]]
    # Save to CSV
    print(f"Saving merged file to: {OUTPUT_PATH}")
    merged.to_csv(OUTPUT_PATH, index=False)
    print("Done.")

main()

Loading files...
Genome rows: 4983
Health rows: 73778
Unique SEQN in health file: 4983
Health unique rows after drop_duplicates: 4983
Merging genome info back into the full health dataframe (left join on SEQN)...
Merged rows: 73778
Rows with genome info present (approx): 73778
Saving merged file to: Final_Dataset.csv
Done.
